In [2]:
# Cell 1 — Setup: load RAW tables into SQLite this time
import pandas as pd
import sqlite3

conn = sqlite3.connect(':memory:')

# Load raw tables 
tables = {
    'orders': pd.read_csv('../data/cleaned/orders_clean.csv'),
    'order_items': pd.read_csv('../data/raw/olist_order_items_dataset.csv'),
    'customers': pd.read_csv('../data/raw/olist_customers_dataset.csv'),
    'products': pd.read_csv('../data/cleaned/products_clean.csv'),
    'sellers': pd.read_csv('../data/raw/olist_sellers_dataset.csv'),
    'payments': pd.read_csv('../data/raw/olist_order_payments_dataset.csv'),
    'reviews': pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')
}

for name, df in tables.items():
    df.to_sql(name, conn, if_exists='replace', index=False)
    print(f"{name}: {len(df):,} rows")

orders: 96,478 rows
order_items: 112,650 rows
customers: 99,441 rows
products: 32,951 rows
sellers: 3,095 rows
payments: 103,886 rows
reviews: 99,224 rows


In [3]:
# Cell 2 — JOIN Query 1
# Business Q: What is total revenue and avg review score per seller state?
# Requires: orders → order_items → sellers + reviews

result = pd.read_sql_query("""
    SELECT 
        s.seller_state,
        COUNT(DISTINCT o.order_id) as total_orders,
        ROUND(SUM(oi.price), 2) as total_revenue,
        ROUND(AVG(r.review_score), 2) as avg_review
    FROM orders o
    INNER JOIN order_items oi ON o.order_id = oi.order_id
    INNER JOIN sellers s ON oi.seller_id = s.seller_id
    LEFT JOIN reviews r ON o.order_id = r.order_id
    GROUP BY s.seller_state
    ORDER BY total_revenue DESC
    LIMIT 10
""", conn)
print("Q1: Revenue and satisfaction by seller state")
print(result)

Q1: Revenue and satisfaction by seller state
  seller_state  total_orders  total_revenue  avg_review
0           SP         68641     8548955.78        4.05
1           PR          7512     1240516.38        4.13
2           MG          7735      981274.22        4.16
3           RJ          4227      823390.38        4.17
4           SC          3603      615030.19        4.13
5           RS          1962      375865.84        4.25
6           BA           550      278046.51        4.16
7           DF           808       94912.19        4.07
8           PE           403       91164.15        4.15
9           GO           451       64886.49        4.31


In [4]:
# Cell 3 — JOIN Query 2
# Business Q: Which product categories have the highest late delivery rate?
# This connects to our Day 4 finding about late orders + high order value

result = pd.read_sql_query("""
    SELECT 
        p.product_category_name_english,
        COUNT(DISTINCT o.order_id) as total_orders,
        SUM(CASE WHEN o.delivery_delay_days > 0 THEN 1 ELSE 0 END) 
            as late_orders,
        ROUND(SUM(CASE WHEN o.delivery_delay_days > 0 THEN 1 ELSE 0 END) 
            * 100.0 / COUNT(DISTINCT o.order_id), 1) as late_pct,
        ROUND(AVG(oi.price), 2) as avg_price
    FROM orders o
    INNER JOIN order_items oi ON o.order_id = oi.order_id
    INNER JOIN products p ON oi.product_id = p.product_id
    WHERE p.product_category_name_english != 'unknown'
    GROUP BY p.product_category_name_english
    HAVING total_orders > 200
    ORDER BY late_pct DESC
    LIMIT 10
""", conn)
print("Q2: Late delivery rate by product category")
print(result)

Q2: Late delivery rate by product category
  product_category_name_english  total_orders  late_orders  late_pct  \
0                         audio           348           42      12.1   
1              office_furniture          1254          133      10.6   
2                  home_confort           392           40      10.2   
3               furniture_decor          6307          574       9.1   
4     construction_tools_lights           242           22       9.1   
5         furniture_living_room           414           35       8.5   
6                 health_beauty          8647          716       8.3   
7                bed_bath_table          9272          770       8.3   
8                   electronics          2517          207       8.2   
9               books_technical           256           21       8.2   

   avg_price  
0     139.70  
1     160.76  
2     135.22  
3      87.25  
4     132.75  
5     136.06  
6     130.28  
7      93.44  
8      56.81  
9      71.11  

In [5]:
# Cell 4 — JOIN Query 3
# Business Q: Which sellers have the best review scores with high volume?
# Minimum 50 orders to filter out noise (like HAVING lesson from Day 4)

result = pd.read_sql_query("""
    SELECT 
        oi.seller_id,
        s.seller_state,
        COUNT(DISTINCT o.order_id) as total_orders,
        ROUND(AVG(r.review_score), 2) as avg_review,
        ROUND(SUM(oi.price), 2) as total_revenue
    FROM orders o
    INNER JOIN order_items oi ON o.order_id = oi.order_id
    INNER JOIN sellers s ON oi.seller_id = s.seller_id
    LEFT JOIN reviews r ON o.order_id = r.order_id
    GROUP BY oi.seller_id, s.seller_state
    HAVING total_orders >= 50
    ORDER BY avg_review DESC, total_orders DESC
    LIMIT 10
""", conn)
print("Q3: Top sellers by review score (min 50 orders)")
print(result)

Q3: Top sellers by review score (min 50 orders)
                          seller_id seller_state  total_orders  avg_review  \
0  d9bd94811c3338dceb4181f3dbc0c73e           SP            54        4.82   
1  d13e50eaa47b4cbe9eb81465865d8cfc           SP            66        4.81   
2  d566c37fa119d5e66c4e9052e83ee4ea           SP            65        4.72   
3  376a891762bbdecbc02b4b6adec3fdda           GO            57        4.67   
4  ac3508719a1d8f5b7614b798f70af136           RS           101        4.64   
5  d9a84e1403de8da0c3aa531d6d108ba6           SP            53        4.62   
6  080199a181c46c657dc5aa235411be3b           SP            79        4.61   
7  516e7738bd8f735ac19a010ee5450d8d           RJ            74        4.61   
8  5b925e1d006e9476d738aa200751b73b           SP            63        4.61   
9  116ccb1a1604bc88e4d234a8c23f33de           SP            61        4.60   

   total_revenue  
0        6911.02  
1        7073.05  
2        5145.70  
3        6406.23 

In [6]:
# Cell 5 — Subquery 1 
# customer_state comes from customers table, not orders
# So we join first, then run the subquery

result = pd.read_sql_query("""
    SELECT 
        c.customer_state,
        COUNT(DISTINCT o.order_id) as total_orders
    FROM orders o
    INNER JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY c.customer_state
    HAVING total_orders > (
        SELECT AVG(state_orders)
        FROM (
            SELECT COUNT(DISTINCT o2.order_id) as state_orders
            FROM orders o2
            INNER JOIN customers c2 ON o2.customer_id = c2.customer_id
            GROUP BY c2.customer_state
        )
    )
    ORDER BY total_orders DESC
""", conn)
print("Q4: States with above-average order counts")
print(result)

Q4: States with above-average order counts
  customer_state  total_orders
0             SP         40501
1             RJ         12350
2             MG         11354
3             RS          5345
4             PR          4923


In [7]:
# Cell 6 — Subquery 2 (practical, interview-style)
# Business Q: Find orders where price is above the average price 
# for that specific product category
# This is a correlated subquery — slightly advanced but very common

result = pd.read_sql_query("""
    SELECT 
        o.order_id,
        p.product_category_name_english,
        oi.price,
        ROUND((
            SELECT AVG(oi2.price) 
            FROM order_items oi2
            INNER JOIN products p2 ON oi2.product_id = p2.product_id
            WHERE p2.product_category_name_english = 
                  p.product_category_name_english
        ), 2) as category_avg_price,
        ROUND(oi.price - (
            SELECT AVG(oi2.price) 
            FROM order_items oi2
            INNER JOIN products p2 ON oi2.product_id = p2.product_id
            WHERE p2.product_category_name_english = 
                  p.product_category_name_english
        ), 2) as price_vs_category_avg
    FROM orders o
    INNER JOIN order_items oi ON o.order_id = oi.order_id
    INNER JOIN products p ON oi.product_id = p.product_id
    WHERE oi.price > 500
    ORDER BY price_vs_category_avg DESC
    LIMIT 10
""", conn)
print("Q5: High-price orders vs their category average")
print(result)

Q5: High-price orders vs their category average
                           order_id product_category_name_english    price  \
0  0812eb902a67711a1cb742b3cdaa65ae                    housewares  6735.00   
1  f5136e38d1a14a4dbd87dff67da82701                           art  6499.00   
2  fefacc66af859508bf1a7934eab1e97f                     computers  6729.00   
3  a96610ab360d42a2e5335a3998b4718a              small_appliances  4799.00   
4  199af31afc78c699f0dbf71fb178d4d4              small_appliances  4690.00   
5  426a9742b533fc6fed17d1fd6d143d7e           musical_instruments  4399.87   
6  68101694e5c5dc7330c91e1bbc36214f                consoles_games  4099.99   
7  b239ca7cd485940b31882363b52e6674                sports_leisure  4059.00   
8  9a3966c23190dbdbaabed08e8429c006                       unknown  3980.00   
9  41b7766bb1df487d17fb9725b78ff509                  garden_tools  3930.00   

   category_avg_price  price_vs_category_avg  
0               90.79                6644.21  

In [8]:
# Cell 7 — RANK(): Top seller per state
# Business Q: Who is the #1 revenue seller in each Brazilian state?

result = pd.read_sql_query("""
    WITH seller_revenue AS (
        SELECT 
            s.seller_state,
            oi.seller_id,
            ROUND(SUM(oi.price), 2) as revenue,
            COUNT(DISTINCT o.order_id) as total_orders,
            RANK() OVER (
                PARTITION BY s.seller_state 
                ORDER BY SUM(oi.price) DESC
            ) as revenue_rank
        FROM orders o
        INNER JOIN order_items oi ON o.order_id = oi.order_id
        INNER JOIN sellers s ON oi.seller_id = s.seller_id
        GROUP BY s.seller_state, oi.seller_id
    )
    SELECT * FROM seller_revenue
    WHERE revenue_rank = 1
    ORDER BY revenue DESC
    LIMIT 10
""", conn)
print("Q6: Top revenue seller in each state")
print(result)

Q6: Top revenue seller in each state
  seller_state                         seller_id    revenue  total_orders  \
0           SP  4869f7a5dfa277a7dca6462dcf3b52b2  226987.93          1124   
1           BA  53243585a1d6dc2643021fd1853d8905  217940.44           348   
2           RJ  46dc3b2cc0980fb8ec44634e21d2718e  122811.38           503   
3           MG  a1043bafd471dff536d0c462352beb48   99309.23           702   
4           PR  ccc4bbb5f32a6ab2b7066a4130f114e3   72926.72           184   
5           SC  04308b1ee57b6625f47df1d56f00eedf   58991.80            92   
6           PE  de722cd6dad950a92b7d4f82673f8833   55126.30           337   
7           MA  06a2c3af7b3aee5d69171b0e14f0ee87   36097.98           389   
8           RS  87142160b41353c4e5fca2360caf6f92   31095.98           305   
9           ES  001cca7ae9ae17fb1caed9dfb1094831   24487.03           195   

   revenue_rank  
0             1  
1             1  
2             1  
3             1  
4             1  
5      

In [9]:
# Cell 8 — LAG(): Month-over-month revenue growth
# Business Q: How did revenue grow month by month? What was the best month?

result = pd.read_sql_query("""
    WITH monthly_revenue AS (
        SELECT 
            strftime('%Y-%m', order_purchase_timestamp) as month,
            ROUND(SUM(oi.price), 2) as revenue
        FROM orders o
        INNER JOIN order_items oi ON o.order_id = oi.order_id
        GROUP BY strftime('%Y-%m', order_purchase_timestamp)
    )
    SELECT 
        month,
        revenue,
        LAG(revenue) OVER (ORDER BY month) as prev_month_revenue,
        ROUND(
            (revenue - LAG(revenue) OVER (ORDER BY month)) 
            * 100.0 / LAG(revenue) OVER (ORDER BY month), 
        1) as mom_growth_pct
    FROM monthly_revenue
    ORDER BY month
""", conn)
print("Q7: Month-over-month revenue growth")
print(result)

Q7: Month-over-month revenue growth
      month    revenue  prev_month_revenue  mom_growth_pct
0   2016-09     134.97                 NaN             NaN
1   2016-10   40325.11              134.97         29777.1
2   2016-12      10.90            40325.11          -100.0
3   2017-01  111798.36               10.90       1025573.0
4   2017-02  234223.40           111798.36           109.5
5   2017-03  359198.85           234223.40            53.4
6   2017-04  340669.68           359198.85            -5.2
7   2017-05  489338.25           340669.68            43.6
8   2017-06  421923.37           489338.25           -13.8
9   2017-07  481604.52           421923.37            14.1
10  2017-08  554699.70           481604.52            15.2
11  2017-09  607399.67           554699.70             9.5
12  2017-10  648247.65           607399.67             6.7
13  2017-11  987765.37           648247.65            52.4
14  2017-12  726033.19           987765.37           -26.5
15  2018-01  924645.

In [10]:
# Cell 9 — Running Total: Cumulative revenue over time
# Business Q: How long did it take to reach 50% of total revenue?

result = pd.read_sql_query("""
    WITH monthly_revenue AS (
        SELECT 
            strftime('%Y-%m', order_purchase_timestamp) as month,
            ROUND(SUM(oi.price), 2) as monthly_rev
        FROM orders o
        INNER JOIN order_items oi ON o.order_id = oi.order_id
        GROUP BY strftime('%Y-%m', order_purchase_timestamp)
    ),
    total AS (
        SELECT SUM(monthly_rev) as total_rev FROM monthly_revenue
    )
    SELECT 
        m.month,
        m.monthly_rev,
        ROUND(SUM(m.monthly_rev) OVER (ORDER BY m.month), 2) 
            as cumulative_revenue,
        ROUND(SUM(m.monthly_rev) OVER (ORDER BY m.month) 
            * 100.0 / t.total_rev, 1) as cumulative_pct
    FROM monthly_revenue m, total t
    ORDER BY m.month
""", conn)
print("Q8: Cumulative revenue % over time")
print(result)

Q8: Cumulative revenue % over time
      month  monthly_rev  cumulative_revenue  cumulative_pct
0   2016-09       134.97              134.97             0.0
1   2016-10     40325.11            40460.08             0.3
2   2016-12        10.90            40470.98             0.3
3   2017-01    111798.36           152269.34             1.2
4   2017-02    234223.40           386492.74             2.9
5   2017-03    359198.85           745691.59             5.6
6   2017-04    340669.68          1086361.27             8.2
7   2017-05    489338.25          1575699.52            11.9
8   2017-06    421923.37          1997622.89            15.1
9   2017-07    481604.52          2479227.41            18.8
10  2017-08    554699.70          3033927.11            22.9
11  2017-09    607399.67          3641326.78            27.5
12  2017-10    648247.65          4289574.43            32.4
13  2017-11    987765.37          5277339.80            39.9
14  2017-12    726033.19          6003372.99      